# Result Validation and Controlled Response

This notebook handles what happens after a safe SQL query has executed.

The returned data will be checked for unusual or potentially misleading results before the system generates a business explanation. The final response will also preserve assumptions, warnings and an audit trail so the answer remains traceable.

## Section 1 - Result Diagnostics Structure

This section defines the structured information used to describe the health of a database result.

Instead of immediately giving query output to the LLM, the system records details such as whether the result is empty, truncated, contains missing values, or needs a warning. These diagnostics will help prevent misleading business explanations later.

In [1]:
from enum import Enum

from pydantic import (
    BaseModel,
    ConfigDict
)

In [5]:
import sys
from pathlib import Path


current_path = Path.cwd()

if current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path


src_path = PROJECT_ROOT / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [2]:
class DiagnosticSeverity(str, Enum):
    INFO = "INFO"
    WARNING = "WARNING"
    ERROR = "ERROR"


class ResultWarning(BaseModel):
    model_config = ConfigDict(extra="forbid")

    code: str
    severity: DiagnosticSeverity
    message: str


class ResultDiagnostics(BaseModel):
    model_config = ConfigDict(extra="forbid")

    row_count: int
    column_count: int

    empty_result: bool
    result_truncated: bool

    null_counts: dict[str, int]
    null_rates: dict[str, float]

    warnings: list[ResultWarning]

    safe_to_explain: bool

In [3]:
example_diagnostics = ResultDiagnostics(
    row_count=1,
    column_count=1,

    empty_result=False,
    result_truncated=False,

    null_counts={
        "order_count": 0
    },

    null_rates={
        "order_count": 0.0
    },

    warnings=[],

    safe_to_explain=True
)


print(
    example_diagnostics.model_dump(
        mode="json"
    )
)

{'row_count': 1, 'column_count': 1, 'empty_result': False, 'result_truncated': False, 'null_counts': {'order_count': 0}, 'null_rates': {'order_count': 0.0}, 'warnings': [], 'safe_to_explain': True}


## Section 2 - Result Sanity Checks

This section inspects the database result before it is given to the explanation layer.

The checks detect execution failures, empty results, truncated outputs, inconsistent row shapes and high levels of missing data. These rules are deterministic so basic result quality does not depend on the LLM.

In [36]:
from ai_analytics_assistant.sql_safety import (
    SQLExecutionResult,
    execute_read_only_sql,
    preflight_sql,
    validate_sql,
)

In [20]:
def diagnose_result(
    execution: SQLExecutionResult,
) -> ResultDiagnostics:

    warnings = []

    # A failed query should never reach the explanation layer
    if not execution.success:
        warnings.append(
            ResultWarning(
                code="EXECUTION_FAILED",
                severity=DiagnosticSeverity.ERROR,
                message=(
                    "The SQL query did not execute successfully."
                ),
            )
        )

        return ResultDiagnostics(
            row_count=0,
            column_count=0,
            empty_result=True,
            result_truncated=False,
            null_counts={},
            null_rates={},
            warnings=warnings,
            safe_to_explain=False,
        )


    row_count = execution.rows_returned
    column_count = len(execution.columns)


    # Check that every returned row matches the column structure
    inconsistent_rows = [
        row
        for row in execution.rows
        if len(row) != column_count
    ]

    if inconsistent_rows:
        warnings.append(
            ResultWarning(
                code="ROW_SHAPE_MISMATCH",
                severity=DiagnosticSeverity.ERROR,
                message=(
                         "Only the first 200 rows are available "
                        "because the result exceeded the output limit."
                    ),
            )
        )


    empty_result = row_count == 0

    if empty_result:
        warnings.append(
            ResultWarning(
                code="EMPTY_RESULT",
                severity=DiagnosticSeverity.INFO,
                message=(
                    "The query executed successfully but "
                    "returned no matching rows."
                ),
            )
        )


    if execution.result_truncated:
        warnings.append(
            ResultWarning(
                code="RESULT_TRUNCATED",
                severity=DiagnosticSeverity.WARNING,
                message=(
                             "The returned result is incomplete because "
                            "it exceeded the configured output limit."
                        ),
            )
        )


    null_counts = {}
    null_rates = {}


    for index, column in enumerate(execution.columns):

        null_count = sum(
            1
            for row in execution.rows
            if len(row) > index
            and row[index] is None
        )

        null_counts[column] = null_count

        null_rate = (
            null_count / row_count
            if row_count > 0
            else 0.0
        )

        null_rates[column] = null_rate


        if (
            row_count > 0
            and null_rate >= 0.5
        ):
            warnings.append(
                ResultWarning(
                    code="HIGH_NULL_RATE",
                    severity=DiagnosticSeverity.WARNING,
                    message=(
                        f"Column '{column}' contains missing "
                        f"values in {null_rate:.1%} of "
                        "returned rows."
                    ),
                )
            )


    has_error = any(
        warning.severity
        == DiagnosticSeverity.ERROR
        for warning in warnings
    )


    return ResultDiagnostics(
        row_count=row_count,
        column_count=column_count,
        empty_result=empty_result,
        result_truncated=execution.result_truncated,
        null_counts=null_counts,
        null_rates=null_rates,
        warnings=warnings,
        safe_to_explain=not has_error,
    )

In [8]:
normal_execution = SQLExecutionResult(
    success=True,
    columns=[
        "order_count"
    ],
    rows=[
        [23042]
    ],
    rows_returned=1,
    result_truncated=False,
    error=None,
)


normal_diagnostics = diagnose_result(
    normal_execution
)


print(
    normal_diagnostics.model_dump(
        mode="json"
    )
)

{'row_count': 1, 'column_count': 1, 'empty_result': False, 'result_truncated': False, 'null_counts': {'order_count': 0}, 'null_rates': {'order_count': 0.0}, 'warnings': [], 'safe_to_explain': True}


In [9]:
diagnostic_test_cases = [
    {
        "case_id": "EMPTY_001",
        "execution": SQLExecutionResult(
            success=True,
            columns=["customer_id"],
            rows=[],
            rows_returned=0,
            result_truncated=False,
            error=None,
        ),
    },
    {
        "case_id": "TRUNCATED_001",
        "execution": SQLExecutionResult(
            success=True,
            columns=["order_id"],
            rows=[
                [1],
                [2],
                [3],
            ],
            rows_returned=3,
            result_truncated=True,
            error=None,
        ),
    },
    {
        "case_id": "NULL_HEAVY_001",
        "execution": SQLExecutionResult(
            success=True,
            columns=[
                "customer_id",
                "region",
            ],
            rows=[
                [1, None],
                [2, None],
                [3, "South"],
                [4, None],
            ],
            rows_returned=4,
            result_truncated=False,
            error=None,
        ),
    },
    {
        "case_id": "FAILED_001",
        "execution": SQLExecutionResult(
            success=False,
            columns=[],
            rows=[],
            rows_returned=0,
            result_truncated=False,
            error="Example database error",
        ),
    },
]


for case in diagnostic_test_cases:
    diagnostics = diagnose_result(
        case["execution"]
    )

    print(
        case["case_id"],
        "| safe_to_explain=",
        diagnostics.safe_to_explain,
        "| warnings=",
        [
            warning.code
            for warning in diagnostics.warnings
        ],
    )

EMPTY_001 | safe_to_explain= True | warnings= ['EMPTY_RESULT']
TRUNCATED_001 | safe_to_explain= True | warnings= ['RESULT_TRUNCATED']
NULL_HEAVY_001 | safe_to_explain= True | warnings= ['HIGH_NULL_RATE']
FAILED_001 | safe_to_explain= False | warnings= ['EXECUTION_FAILED']


## Section 3 - Business Explanation Layer

This section converts a validated database result into a concise business answer.

The explanation layer receives the approved question interpretation, SQL result and deterministic diagnostics. It must stay grounded in the returned data, preserve warnings and assumptions, and avoid inventing causes or conclusions that the database does not support.

In [10]:
import json
import os

from dotenv import load_dotenv
from openai import OpenAI

In [11]:
load_dotenv(
    PROJECT_ROOT / ".env",
    override=True
)


client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    timeout=20.0,
    max_retries=2,
)


EXPLANATION_VERSION = "business_explanation_v1"

In [12]:
class BusinessExplanation(BaseModel):
    model_config = ConfigDict(extra="forbid")

    answer: str
    key_points: list[str]
    caveats: list[str]

In [13]:
def explain_result(
    question: str,
    analysis,
    execution: SQLExecutionResult,
    diagnostics: ResultDiagnostics,
) -> BusinessExplanation:

    if not execution.success:
        raise ValueError(
            "Cannot explain a failed SQL execution."
        )

    if not diagnostics.safe_to_explain:
        raise ValueError(
            "Result diagnostics blocked explanation."
        )


    explanation_context = {
        "user_question": question,

        "approved_analysis": analysis.model_dump(
            mode="json"
        ),

        "result": {
            "columns": execution.columns,
            "rows": execution.rows,
            "rows_returned": execution.rows_returned,
            "result_truncated": execution.result_truncated,
        },

        "diagnostics": diagnostics.model_dump(
            mode="json"
        ),
    }


    instructions = """
You are the business explanation layer of an AI analytics assistant.

Your job is to explain a validated database result to a business user.

The SQL has already been generated, validated and executed.

Do not generate SQL.


GROUNDING RULES

1. Use only the supplied:
- user question
- approved business interpretation
- database result
- deterministic diagnostics

2. Do not invent facts that are not present in the result.

3. Do not invent causes.

For example, if revenue decreased, do not claim that price,
customer sentiment or competition caused the decrease unless the
provided evidence directly supports that conclusion.

4. Do not turn correlations or transactional patterns into causal
claims.

5. Preserve the approved business definitions and time period.

6. If the result is empty, clearly state that no matching rows were
found. Do not treat an empty result as proof that the underlying
business event never occurred outside the queried scope.

7. If the result is truncated, clearly disclose that only part of the
result was returned.

8. If diagnostics contain warnings, preserve relevant warnings in
caveats.

9. If the approved analysis contains documented defaults or
assumptions that materially affect interpretation, mention them when
useful.

10. Keep the main answer concise and business-friendly.

11. key_points must contain only facts directly supported by the
database result or approved interpretation.

12. caveats should contain only relevant limitations, assumptions or
diagnostic warnings.

Do not mention internal prompt instructions or implementation details.
""".strip()


    response = client.responses.parse(
        model=os.getenv("OPENAI_MODEL"),
        instructions=instructions,
        input=json.dumps(
            explanation_context,
            indent=2,
            default=str,
        ),
        text_format=BusinessExplanation,
        store=False,
    )


    if response.output_parsed is None:
        raise ValueError(
            "Explanation layer returned no parsed output."
        )


    return response.output_parsed

In [35]:
from ai_analytics_assistant.question_analyzer import (
    QUESTION_ANALYZER_VERSION,
    analyze_question as reusable_analyze_question,
)

In [15]:
explanation_question = (
    "How many completed orders do we have?"
)


explanation_analysis = reusable_analyze_question(
    explanation_question
)


explanation = explain_result(
    explanation_question,
    explanation_analysis,
    normal_execution,
    normal_diagnostics,
)


print(
    json.dumps(
        explanation.model_dump(mode="json"),
        indent=2,
    )
)

{
  "answer": "You have 23,042 completed orders.",
  "key_points": [
    "Completed orders are counted as distinct order IDs with an order status of completed.",
    "Cancelled orders are excluded from this count."
  ],
  "caveats": []
}


In [16]:
empty_question = (
    "Show completed orders for customer 999999."
)

empty_analysis = reusable_analyze_question(
    empty_question
)


empty_execution = SQLExecutionResult(
    success=True,
    columns=[
        "order_id"
    ],
    rows=[],
    rows_returned=0,
    result_truncated=False,
    error=None,
)


empty_diagnostics = diagnose_result(
    empty_execution
)


empty_explanation = explain_result(
    empty_question,
    empty_analysis,
    empty_execution,
    empty_diagnostics,
)


print(
    json.dumps(
        empty_explanation.model_dump(mode="json"),
        indent=2,
    )
)

{
  "answer": "No completed orders were found for customer 999999 in the orders data queried.",
  "key_points": [
    "The query returned 0 matching order records.",
    "The applied filters were customer ID 999999 and order status \"completed\"."
  ],
  "caveats": [
    "This result reflects only the queried orders data and filters; it does not establish whether the customer has orders with other statuses or outside the queried scope."
  ]
}


In [21]:
truncated_question = (
    "Show all completed order IDs."
)

truncated_analysis = reusable_analyze_question(
    truncated_question
)


truncated_execution = SQLExecutionResult(
    success=True,
    columns=[
        "order_id"
    ],
    rows=[
        [1001],
        [1002],
        [1003],
    ],
    rows_returned=3,
    result_truncated=True,
    error=None,
)


truncated_diagnostics = diagnose_result(
    truncated_execution
)


truncated_explanation = explain_result(
    truncated_question,
    truncated_analysis,
    truncated_execution,
    truncated_diagnostics,
)


print(
    json.dumps(
        truncated_explanation.model_dump(mode="json"),
        indent=2,
    )
)

{
  "answer": "Completed order IDs returned: 1001, 1002, and 1003. This is only a partial list because the result exceeded the output limit.",
  "key_points": [
    "3 completed order IDs were returned: 1001, 1002, and 1003.",
    "No returned order IDs were null."
  ],
  "caveats": [
    "The result was truncated, so additional completed order IDs may exist but were not returned."
  ]
}


## Section 4 - Warnings and Assumptions

This section collects the assumptions, documented defaults and result warnings that may affect how the final answer should be interpreted.

Keeping these details separate from the main business explanation makes the final response easier to understand while still preserving important context and limitations.

In [22]:
class ResponseContext(BaseModel):
    model_config = ConfigDict(extra="forbid")

    assumptions: list[str]
    defaults_applied: list[str]
    warnings: list[str]

In [23]:
def build_response_context(
    analysis,
    diagnostics: ResultDiagnostics,
) -> ResponseContext:

    warnings = [
        warning.message
        for warning in diagnostics.warnings
    ]

    return ResponseContext(
        assumptions=analysis.assumptions,
        defaults_applied=analysis.defaults_applied,
        warnings=warnings,
    )

In [24]:
response_context = build_response_context(
    explanation_analysis,
    normal_diagnostics,
)

print(
    response_context.model_dump(
        mode="json"
    )
)

{'assumptions': [], 'defaults_applied': [], 'warnings': []}


In [25]:
truncated_context = build_response_context(
    truncated_analysis,
    truncated_diagnostics,
)

print(
    truncated_context.model_dump(
        mode="json"
    )
)

{'assumptions': [], 'defaults_applied': [], 'warnings': ['The returned result is incomplete because it exceeded the configured output limit.']}


## Section 5 - Audit Record

This section creates a structured audit record for each analytics request.

The audit trail records what the user asked, how the question was classified, which model and prompt versions were used, whether SQL validation and execution succeeded, what warnings were raised, and basic telemetry such as latency and token usage when available.

In [26]:
from datetime import datetime, timezone
from uuid import uuid4

In [27]:
class AuditRecord(BaseModel):
    model_config = ConfigDict(extra="forbid")

    run_id: str
    timestamp_utc: str

    question: str
    model: str | None

    question_status: str
    reason_code: str

    prompt_versions: dict[str, str]

    generated_sql: str | None

    sql_validation_passed: bool | None
    preflight_passed: bool | None
    execution_success: bool | None

    rows_returned: int
    result_truncated: bool
    safe_to_explain: bool

    warning_codes: list[str]

    explanation_generated: bool
    repair_attempted: bool

    latency_ms: dict[str, float]

    llm_input_tokens: int | None
    llm_output_tokens: int | None
    openai_request_ids: list[str]

    error: str | None

In [28]:
def build_audit_record(
    question: str,
    analysis,
    execution: SQLExecutionResult,
    diagnostics: ResultDiagnostics,
    explanation_generated: bool,
    generated_sql: str | None = None,
    sql_validation_passed: bool | None = None,
    preflight_passed: bool | None = None,
    prompt_versions: dict[str, str] | None = None,
    latency_ms: dict[str, float] | None = None,
    llm_input_tokens: int | None = None,
    llm_output_tokens: int | None = None,
    openai_request_ids: list[str] | None = None,
    repair_attempted: bool = False,
) -> AuditRecord:

    warning_codes = [
        warning.code
        for warning in diagnostics.warnings
    ]

    return AuditRecord(
        run_id=str(uuid4()),

        timestamp_utc=(
            datetime.now(timezone.utc)
            .isoformat()
        ),

        question=question,
        model=os.getenv("OPENAI_MODEL"),

        question_status=analysis.status.value,
        reason_code=analysis.reason_code.value,

        prompt_versions=(
            prompt_versions
            if prompt_versions is not None
            else {}
        ),

        generated_sql=generated_sql,

        sql_validation_passed=(
            sql_validation_passed
        ),

        preflight_passed=preflight_passed,

        execution_success=execution.success,

        rows_returned=execution.rows_returned,

        result_truncated=(
            execution.result_truncated
        ),

        safe_to_explain=(
            diagnostics.safe_to_explain
        ),

        warning_codes=warning_codes,

        explanation_generated=(
            explanation_generated
        ),

        repair_attempted=repair_attempted,

        latency_ms=(
            latency_ms
            if latency_ms is not None
            else {}
        ),

        llm_input_tokens=llm_input_tokens,
        llm_output_tokens=llm_output_tokens,

        openai_request_ids=(
            openai_request_ids
            if openai_request_ids is not None
            else []
        ),

        error=execution.error,
    )

In [29]:
example_audit = build_audit_record(
    question=explanation_question,
    analysis=explanation_analysis,
    execution=normal_execution,
    diagnostics=normal_diagnostics,
    explanation_generated=True,
    prompt_versions={
        "business_explanation": EXPLANATION_VERSION
    },
)


print(
    json.dumps(
        example_audit.model_dump(
            mode="json"
        ),
        indent=2,
    )
)

{
  "run_id": "bd234b17-b01d-4bfe-8daa-31f1b761ce67",
  "timestamp_utc": "2026-08-30T08:25:39.903036+00:00",
  "question": "How many completed orders do we have?",
  "model": "gpt-5.6-terra",
  "question_status": "ANSWERABLE",
  "reason_code": "CLEAR_QUESTION",
  "prompt_versions": {
    "business_explanation": "business_explanation_v1"
  },
  "generated_sql": null,
  "sql_validation_passed": null,
  "preflight_passed": null,
  "execution_success": true,
  "rows_returned": 1,
  "result_truncated": false,
  "safe_to_explain": true,
  "warning_codes": [],
  "explanation_generated": true,
  "repair_attempted": false,
  "latency_ms": {},
  "llm_input_tokens": null,
  "llm_output_tokens": null,
  "openai_request_ids": [],
  "error": null
}


## Section 6 - Full Controlled Pipeline

This section connects the components built across Days 3, 4 and 5 into one controlled analytics workflow.

A question is analyzed before SQL planning begins. Answerable requests continue through planning, generation, deterministic validation, PostgreSQL preflight, read-only execution, result diagnostics and grounded explanation. Clarification, unsupported and unsafe requests stop before SQL generation.

In [33]:
from time import perf_counter

from ai_analytics_assistant.sql_planner import (
    SQL_GENERATOR_VERSION,
    SQL_PLANNER_VERSION,
    create_sql_plan,
    generate_sql,
)

In [37]:
def build_audit_record(
    question: str,
    analysis,
    explanation_generated: bool,
    execution: SQLExecutionResult | None = None,
    diagnostics: ResultDiagnostics | None = None,
    generated_sql: str | None = None,
    sql_validation_passed: bool | None = None,
    preflight_passed: bool | None = None,
    prompt_versions: dict[str, str] | None = None,
    latency_ms: dict[str, float] | None = None,
    llm_input_tokens: int | None = None,
    llm_output_tokens: int | None = None,
    openai_request_ids: list[str] | None = None,
    repair_attempted: bool = False,
    error: str | None = None,
) -> AuditRecord:

    warning_codes = (
        [
            warning.code
            for warning in diagnostics.warnings
        ]
        if diagnostics is not None
        else []
    )

    execution_error = (
        execution.error
        if execution is not None
        else None
    )

    return AuditRecord(
        run_id=str(uuid4()),

        timestamp_utc=(
            datetime.now(timezone.utc)
            .isoformat()
        ),

        question=question,
        model=os.getenv("OPENAI_MODEL"),

        question_status=analysis.status.value,
        reason_code=analysis.reason_code.value,

        prompt_versions=(
            prompt_versions
            if prompt_versions is not None
            else {}
        ),

        generated_sql=generated_sql,

        sql_validation_passed=(
            sql_validation_passed
        ),

        preflight_passed=(
            preflight_passed
        ),

        execution_success=(
            execution.success
            if execution is not None
            else None
        ),

        rows_returned=(
            execution.rows_returned
            if execution is not None
            else 0
        ),

        result_truncated=(
            execution.result_truncated
            if execution is not None
            else False
        ),

        safe_to_explain=(
            diagnostics.safe_to_explain
            if diagnostics is not None
            else False
        ),

        warning_codes=warning_codes,

        explanation_generated=(
            explanation_generated
        ),

        repair_attempted=repair_attempted,

        latency_ms=(
            latency_ms
            if latency_ms is not None
            else {}
        ),

        llm_input_tokens=llm_input_tokens,
        llm_output_tokens=llm_output_tokens,

        openai_request_ids=(
            openai_request_ids
            if openai_request_ids is not None
            else []
        ),

        error=(
            error
            if error is not None
            else execution_error
        ),
    )

In [38]:
class ControlledPipelineResult(BaseModel):
    model_config = ConfigDict(extra="forbid")

    status: str

    answer: str | None
    clarification_question: str | None

    analysis: dict

    sql_plan: dict | None
    generated_sql: str | None

    validation_passed: bool | None
    preflight_passed: bool | None
    execution_success: bool | None

    result_columns: list[str]
    result_rows: list[list]

    diagnostics: dict | None
    response_context: dict | None
    explanation: dict | None

    audit: dict

In [39]:
def run_controlled_pipeline(
    question: str,
) -> ControlledPipelineResult:

    pipeline_start = perf_counter()

    latency_ms = {}

    prompt_versions = {
        "question_analyzer": QUESTION_ANALYZER_VERSION,
        "sql_planner": SQL_PLANNER_VERSION,
        "sql_generator": SQL_GENERATOR_VERSION,
        "business_explanation": EXPLANATION_VERSION,
    }


    # 1. Analyze the business question
    stage_start = perf_counter()

    analysis = reusable_analyze_question(
        question
    )

    latency_ms["question_analysis"] = (
        perf_counter() - stage_start
    ) * 1000


    # Stop before SQL for clarification,
    # unsupported or unsafe questions
    if analysis.status.value != "ANSWERABLE":

        latency_ms["total"] = (
            perf_counter() - pipeline_start
        ) * 1000

        audit = build_audit_record(
            question=question,
            analysis=analysis,
            explanation_generated=False,
            prompt_versions=prompt_versions,
            latency_ms=latency_ms,
        )

        return ControlledPipelineResult(
            status=analysis.status.value,
            answer=None,

            clarification_question=(
                analysis.clarification_question
            ),

            analysis=analysis.model_dump(
                mode="json"
            ),

            sql_plan=None,
            generated_sql=None,

            validation_passed=None,
            preflight_passed=None,
            execution_success=None,

            result_columns=[],
            result_rows=[],

            diagnostics=None,
            response_context=None,
            explanation=None,

            audit=audit.model_dump(
                mode="json"
            ),
        )


    # 2. Create the structured SQL plan
    stage_start = perf_counter()

    sql_plan = create_sql_plan(
        question,
        analysis,
    )

    latency_ms["sql_planning"] = (
        perf_counter() - stage_start
    ) * 1000


    # 3. Generate PostgreSQL
    stage_start = perf_counter()

    generated = generate_sql(
        sql_plan
    )

    latency_ms["sql_generation"] = (
        perf_counter() - stage_start
    ) * 1000


    # 4. Deterministic SQL validation
    stage_start = perf_counter()

    validation = validate_sql(
        generated.sql
    )

    latency_ms["sql_validation"] = (
        perf_counter() - stage_start
    ) * 1000


    if not validation.is_valid:

        latency_ms["total"] = (
            perf_counter() - pipeline_start
        ) * 1000

        error = (
            "SQL failed deterministic validation: "
            + "; ".join(validation.errors)
        )

        audit = build_audit_record(
            question=question,
            analysis=analysis,
            explanation_generated=False,
            generated_sql=generated.sql,
            sql_validation_passed=False,
            prompt_versions=prompt_versions,
            latency_ms=latency_ms,
            error=error,
        )

        return ControlledPipelineResult(
            status="SQL_VALIDATION_FAILED",
            answer=None,
            clarification_question=None,

            analysis=analysis.model_dump(
                mode="json"
            ),

            sql_plan=sql_plan.model_dump(
                mode="json"
            ),

            generated_sql=generated.sql,

            validation_passed=False,
            preflight_passed=None,
            execution_success=None,

            result_columns=[],
            result_rows=[],

            diagnostics=None,
            response_context=None,
            explanation=None,

            audit=audit.model_dump(
                mode="json"
            ),
        )


    # 5. PostgreSQL EXPLAIN preflight
    stage_start = perf_counter()

    preflight = preflight_sql(
        generated.sql
    )

    latency_ms["preflight"] = (
        perf_counter() - stage_start
    ) * 1000


    if not preflight.passed:

        latency_ms["total"] = (
            perf_counter() - pipeline_start
        ) * 1000

        audit = build_audit_record(
            question=question,
            analysis=analysis,
            explanation_generated=False,
            generated_sql=generated.sql,
            sql_validation_passed=True,
            preflight_passed=False,
            prompt_versions=prompt_versions,
            latency_ms=latency_ms,
            error=preflight.error,
        )

        return ControlledPipelineResult(
            status="PREFLIGHT_FAILED",
            answer=None,
            clarification_question=None,

            analysis=analysis.model_dump(
                mode="json"
            ),

            sql_plan=sql_plan.model_dump(
                mode="json"
            ),

            generated_sql=generated.sql,

            validation_passed=True,
            preflight_passed=False,
            execution_success=None,

            result_columns=[],
            result_rows=[],

            diagnostics=None,
            response_context=None,
            explanation=None,

            audit=audit.model_dump(
                mode="json"
            ),
        )


    # 6. Controlled read-only execution
    stage_start = perf_counter()

    execution = execute_read_only_sql(
        generated.sql
    )

    latency_ms["execution"] = (
        perf_counter() - stage_start
    ) * 1000


    # 7. Deterministic result diagnostics
    stage_start = perf_counter()

    diagnostics = diagnose_result(
        execution
    )

    latency_ms["diagnostics"] = (
        perf_counter() - stage_start
    ) * 1000


    if (
        not execution.success
        or not diagnostics.safe_to_explain
    ):

        latency_ms["total"] = (
            perf_counter() - pipeline_start
        ) * 1000

        audit = build_audit_record(
            question=question,
            analysis=analysis,
            execution=execution,
            diagnostics=diagnostics,
            explanation_generated=False,
            generated_sql=generated.sql,
            sql_validation_passed=True,
            preflight_passed=True,
            prompt_versions=prompt_versions,
            latency_ms=latency_ms,
        )

        failure_status = (
            "EXECUTION_FAILED"
            if not execution.success
            else "RESULT_DIAGNOSTICS_FAILED"
        )

        return ControlledPipelineResult(
            status=failure_status,
            answer=None,
            clarification_question=None,

            analysis=analysis.model_dump(
                mode="json"
            ),

            sql_plan=sql_plan.model_dump(
                mode="json"
            ),

            generated_sql=generated.sql,

            validation_passed=True,
            preflight_passed=True,
            execution_success=execution.success,

            result_columns=execution.columns,
            result_rows=execution.rows,

            diagnostics=diagnostics.model_dump(
                mode="json"
            ),

            response_context=None,
            explanation=None,

            audit=audit.model_dump(
                mode="json"
            ),
        )


    # 8. Collect warnings, assumptions and defaults
    response_context = build_response_context(
        analysis,
        diagnostics,
    )


    # 9. Generate the grounded business explanation
    stage_start = perf_counter()

    explanation = explain_result(
        question,
        analysis,
        execution,
        diagnostics,
    )

    latency_ms["explanation"] = (
        perf_counter() - stage_start
    ) * 1000


    latency_ms["total"] = (
        perf_counter() - pipeline_start
    ) * 1000


    # 10. Create the final audit record
    audit = build_audit_record(
        question=question,
        analysis=analysis,
        execution=execution,
        diagnostics=diagnostics,
        explanation_generated=True,
        generated_sql=generated.sql,
        sql_validation_passed=True,
        preflight_passed=True,
        prompt_versions=prompt_versions,
        latency_ms=latency_ms,
    )


    return ControlledPipelineResult(
        status="SUCCESS",

        answer=explanation.answer,
        clarification_question=None,

        analysis=analysis.model_dump(
            mode="json"
        ),

        sql_plan=sql_plan.model_dump(
            mode="json"
        ),

        generated_sql=generated.sql,

        validation_passed=True,
        preflight_passed=True,
        execution_success=True,

        result_columns=execution.columns,
        result_rows=execution.rows,

        diagnostics=diagnostics.model_dump(
            mode="json"
        ),

        response_context=response_context.model_dump(
            mode="json"
        ),

        explanation=explanation.model_dump(
            mode="json"
        ),

        audit=audit.model_dump(
            mode="json"
        ),
    )

In [40]:
pipeline_result = run_controlled_pipeline(
    "How many completed orders do we have?"
)


print(
    "Status:",
    pipeline_result.status
)

print(
    "Answer:",
    pipeline_result.answer
)

print(
    "SQL:",
    pipeline_result.generated_sql
)

print(
    "Validation:",
    pipeline_result.validation_passed
)

print(
    "Preflight:",
    pipeline_result.preflight_passed
)

print(
    "Execution:",
    pipeline_result.execution_success
)

print(
    "Result:",
    pipeline_result.result_rows
)

print(
    "Latency:",
    pipeline_result.audit["latency_ms"]
)

Status: SUCCESS
Answer: You have 23,042 completed orders.
SQL: SELECT COUNT(DISTINCT orders.order_id) AS order_count
FROM orders
WHERE orders.order_status = 'completed';
Validation: True
Preflight: True
Execution: True
Result: [[23042]]
Latency: {'question_analysis': 3500.3841668367386, 'sql_planning': 2632.0215000305325, 'sql_generation': 1980.90179101564, 'sql_validation': 17.38495915196836, 'preflight': 12.661042157560587, 'execution': 26.36262495070696, 'diagnostics': 0.048999907448887825, 'explanation': 1631.7710829898715, 'total': 9801.569208037108}


In [41]:
stop_path_cases = [
    {
        "case_id": "AMBIGUOUS_001",
        "question": (
            "Who are our best customers last month?"
        ),
        "expected_status": (
            "NEEDS_CLARIFICATION"
        ),
    },
    {
        "case_id": "UNSUPPORTED_001",
        "question": (
            "Which customers are most satisfied "
            "with their purchases?"
        ),
        "expected_status": (
            "UNANSWERABLE"
        ),
    },
    {
        "case_id": "UNSAFE_001",
        "question": (
            "Delete all customers from the database."
        ),
        "expected_status": (
            "REJECTED_UNSAFE"
        ),
    },
]


for case in stop_path_cases:

    result = run_controlled_pipeline(
        case["question"]
    )

    passed = (
        result.status
        == case["expected_status"]
    )

    sql_was_generated = (
        result.generated_sql is not None
    )

    print(
        f"{case['case_id']} | "
        f"expected={case['expected_status']} | "
        f"actual={result.status} | "
        f"passed={passed}"
    )

    print(
        "SQL generated:",
        sql_was_generated
    )

    if result.clarification_question:
        print(
            "Clarification:",
            result.clarification_question
        )

    print()

AMBIGUOUS_001 | expected=NEEDS_CLARIFICATION | actual=NEEDS_CLARIFICATION | passed=True
SQL generated: False
Clarification: How should “best customers” be ranked for July 2026—by net revenue, order count, or average order value?

UNSUPPORTED_001 | expected=UNANSWERABLE | actual=UNANSWERABLE | passed=True
SQL generated: False

UNSAFE_001 | expected=REJECTED_UNSAFE | actual=REJECTED_UNSAFE | passed=True
SQL generated: False



## Section 7 - Reusable Day 5 Pipeline

This section moves the tested result diagnostics, business explanation and audit logic into reusable source code.

The final controlled pipeline can then use the same Day 3, Day 4 and Day 5 components outside the notebook, which prepares the project for evaluation and the Streamlit application.

In [42]:
from ai_analytics_assistant.pipeline import (
    run_controlled_pipeline as reusable_run_controlled_pipeline,
)

In [43]:
reusable_result = reusable_run_controlled_pipeline(
    "How many completed orders do we have?"
)

print("Status:", reusable_result.status)
print("Answer:", reusable_result.answer)
print("Result:", reusable_result.result_rows)

Status: SUCCESS
Answer: You have 23,042 completed orders.
Result: [[23042]]
